# Dry Bean Classification

For Part II we try the same k-nearest-neighbors approach on the Dry Bean dataset, where the goal is to tell 7 types of dry beans apart from shape measurements taken from photos of each bean.

### Loading the data
The dataset has 13,611 beans, each described by 16 numerical measurements (size and shape) and a `Class` column with the bean type. There are no missing values, so no cleaning is needed.

In [2]:
%pip install openpyxl


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_excel('data/Dry_Bean_Dataset.xlsx')

In [4]:
df.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [5]:
df['Class'].value_counts()

Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

### Preparing the features
All 16 attributes are numbers, so there's nothing to one-hot encode this time. Their scales are very different though (`Area` is in the tens of thousands while `ShapeFactor2` is around 0.002), so standardizing matters even more here.

We split into train and test first, using `stratify` so each bean type shows up in the same proportions in both, and standardize using only the training set's mean and std.

In [6]:
attributes = [c for c in df.columns if c != 'Class']

x_train, x_test, y_train, y_test = train_test_split(df[attributes], df['Class'], test_size=0.2, random_state=63, stratify=df['Class'])

# Standardize features using only the training set's mean and std
for feature in attributes:
    mean, std = x_train[feature].mean(), x_train[feature].std()
    x_train[feature] = (x_train[feature] - mean) / std
    x_test[feature] = (x_test[feature] - mean) / std

### Our own kNN
Same class as in Part I, with one change: instead of counting disease vs healthy, the vote now counts how many neighbors belong to each of the 7 bean types and picks the most common one. If two types tie, the one whose neighbor is closest wins, since neighbors are counted from nearest to farthest.

In [7]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import NearestNeighbors

class MyKNN(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.fit_ = NearestNeighbors(n_neighbors=self.n_neighbors).fit(np.asarray(X))
        self.y_ = np.asarray(y)
        self.classes_ = np.unique(self.y_)
        return self

    def predict(self, X):
        distances, indices = self.fit_.kneighbors(np.asarray(X))
        predictions = []
        for neighbors in indices:
            votes = {}
            for index in neighbors:
                votes[self.y_[index]] = votes.get(self.y_[index], 0) + 1
            predictions.append(max(votes, key=votes.get))
        return np.array(predictions)

### Baseline: all 16 attributes
Like in Part I, we start with every attribute and let `GridSearchCV` try each odd k from 1 to 41 with 10-fold cross-validation. Since there are 7 classes, we score with macro F1: the F1 score of each bean type, averaged, so the small Bombay class counts as much as the big Dermason class.

In [8]:
from sklearn.model_selection import GridSearchCV, KFold

kfold = KFold(n_splits=10, shuffle=True, random_state=63)
nn = MyKNN()
params = {'n_neighbors': list(range(1, 42, 2))}

features = attributes
clf = GridSearchCV(nn, params, cv=kfold, scoring='f1_macro', n_jobs=-1)
clf.fit(x_train[features], y_train)
print(f"Results for all {len(attributes)} attributes: Best params: {clf.best_params_}, Best score: {clf.best_score_}")

Results for all 16 attributes: Best params: {'n_neighbors': 9}, Best score: 0.9369945305936694


### Picking the attributes
Many of these attributes measure nearly the same thing (for example `Area` and `ConvexArea` are almost identical), so using all of them counts "size" several times over. We use the same forward selection as in Part I: add one attribute at a time, keeping whichever helps macro F1 the most, and pick the step with the highest score. This takes a while on 13,000+ beans.

In [9]:
selected = []
remaining = list(attributes)
history = []
while remaining:
    best_score, best_attribute, best_k = 0, None, None
    for attribute in remaining:
        features = selected + [attribute]
        clf = GridSearchCV(nn, params, cv=kfold, scoring='f1_macro', n_jobs=-1)
        clf.fit(x_train[features], y_train)
        if clf.best_score_ > best_score:
            best_score, best_attribute, best_k = clf.best_score_, attribute, clf.best_params_['n_neighbors']
    selected.append(best_attribute)
    remaining.remove(best_attribute)
    history.append((best_score, best_k, list(selected)))
    print(f"Results for {len(selected)} attributes: Best k: {best_k}, Best score: {best_score:.4f}, Attributes: {selected}")

best_score, k, chosen = max(history, key=lambda h: h[0])
features = chosen
print(f"\nChosen: k = {k}, attributes = {chosen}, cross-validated F1 = {best_score:.4f}")

Results for 1 attributes: Best k: 13, Best score: 0.6151, Attributes: ['Perimeter']
Results for 2 attributes: Best k: 31, Best score: 0.9032, Attributes: ['Perimeter', 'ShapeFactor2']
Results for 3 attributes: Best k: 13, Best score: 0.9265, Attributes: ['Perimeter', 'ShapeFactor2', 'roundness']
Results for 4 attributes: Best k: 23, Best score: 0.9294, Attributes: ['Perimeter', 'ShapeFactor2', 'roundness', 'Compactness']
Results for 5 attributes: Best k: 17, Best score: 0.9326, Attributes: ['Perimeter', 'ShapeFactor2', 'roundness', 'Compactness', 'ShapeFactor4']
Results for 6 attributes: Best k: 9, Best score: 0.9360, Attributes: ['Perimeter', 'ShapeFactor2', 'roundness', 'Compactness', 'ShapeFactor4', 'ShapeFactor1']
Results for 7 attributes: Best k: 7, Best score: 0.9385, Attributes: ['Perimeter', 'ShapeFactor2', 'roundness', 'Compactness', 'ShapeFactor4', 'ShapeFactor1', 'Solidity']
Results for 8 attributes: Best k: 7, Best score: 0.9399, Attributes: ['Perimeter', 'ShapeFactor2', 'r

### Final results on the test set
Same as Part I: 10-fold cross-validation on the training set gives us 10 slightly different models, and we test each one on the held-out test set. Precision, recall and F1 are worked out for each bean type and then averaged (macro).

In [10]:
from sklearn import metrics

kfold = KFold(n_splits=10, shuffle=True, random_state=63)
precision_list = []
recall_list = []
f1_score_list = []
for i, (train_index, val_index) in enumerate(kfold.split(x_train, y_train)):
    print(f"Fold {i+1}:")
    x_df = x_train.iloc[train_index]
    x = x_df[features].values
    nn = NearestNeighbors(n_neighbors=k)
    fit = nn.fit(x)

    distances, indices = fit.kneighbors(x_test[features].values)
    predictions = []
    for neighbors in indices:
        votes = {}
        for index in neighbors:
            label = y_train.iloc[train_index[index]]
            votes[label] = votes.get(label, 0) + 1
        predictions.append(max(votes, key=votes.get))

    precision = metrics.precision_score(y_test, predictions, average='macro')
    recall = metrics.recall_score(y_test, predictions, average='macro')
    f1_score = metrics.f1_score(y_test, predictions, average='macro')
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}")

    precision_list.append(precision)
    recall_list.append(recall)
    f1_score_list.append(f1_score)

    print()

print(f"Average Precision: {sum(precision_list)/len(precision_list):.4f}")
print(f"Average Recall: {sum(recall_list)/len(recall_list):.4f}")
print(f"Average F1 Score: {sum(f1_score_list)/len(f1_score_list):.4f}")

Fold 1:
Precision: 0.9347, Recall: 0.9334, F1 Score: 0.9340

Fold 2:
Precision: 0.9348, Recall: 0.9338, F1 Score: 0.9343

Fold 3:
Precision: 0.9333, Recall: 0.9310, F1 Score: 0.9321

Fold 4:
Precision: 0.9343, Recall: 0.9326, F1 Score: 0.9334

Fold 5:
Precision: 0.9332, Recall: 0.9316, F1 Score: 0.9324

Fold 6:
Precision: 0.9351, Recall: 0.9331, F1 Score: 0.9341

Fold 7:
Precision: 0.9349, Recall: 0.9338, F1 Score: 0.9343

Fold 8:
Precision: 0.9380, Recall: 0.9363, F1 Score: 0.9371

Fold 9:
Precision: 0.9374, Recall: 0.9359, F1 Score: 0.9366

Fold 10:
Precision: 0.9347, Recall: 0.9325, F1 Score: 0.9335

Average Precision: 0.9351
Average Recall: 0.9334
Average F1 Score: 0.9342


### Summary
- **k = 7**
- **Attributes (10):** Perimeter, ShapeFactor2, roundness, Compactness, ShapeFactor4, ShapeFactor1, Solidity, EquivDiameter, ShapeFactor3, MinorAxisLength
- **Mean macro F1 on the test set: 0.9342** (average precision 0.9351, average recall 0.9334)

kNN works really well on this dataset, much better than on the heart disease data. Even with all 16 attributes it reached a cross-validated F1 of 0.937, and our 10 selected attributes only nudged that up to 0.940. Unlike Part I, picking attributes barely mattered here: just 3 attributes (Perimeter, ShapeFactor2, roundness) already got 0.927, and after about 7 the score stays flat. That makes sense, since many of the bean measurements describe the same size and shape information in slightly different ways. The 10 test scores are very close to each other because the test set is large (2,723 beans) and every fold model is trained on almost the same data.